# Токсичные комментарии: Spark ML + Kafka + Structured Streaming

Учебная итоговая работа. Обучаю модель на исторических комментариях, затем применяю её к отдельной выборке, поступающей через Kafka. Считаю предсказания по минутным окнам и строю график из memory sink.

Источник: [Google Civil Comments](https://huggingface.co/datasets/google/civil_comments), CC0. В условии нет прямой ссылки на CSV, поэтому использован открытый набор с нужными полями. Из исходного validation-файла делаю собственное учебное разделение, это **не официальный benchmark**. Метка 1: доля оценок токсичности ≥ 0.5. Метки тестовой части нужны только для оценки, модель их не получает как признаки.

In [1]:
import os, json, hashlib, time, uuid
from pathlib import Path
from datetime import datetime, timedelta, timezone
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from kafka import KafkaProducer
from kafka.admin import KafkaAdminClient, NewTopic

Path('work').mkdir(exist_ok=True)
Path('evidence').mkdir(exist_ok=True)
spark = (SparkSession.builder.master('local[2]').appName('toxic-comments-course')
    .config('spark.driver.memory','512m')
    .config('spark.sql.shuffle.partitions','2')
    .config('spark.sql.session.timeZone','UTC')
    .config('spark.ui.enabled','false')
    .config('spark.jars.packages','org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.6')
    .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark:', spark.version)

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a04343d3-1571-4b49-8874-8f2f760e2ff4;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.6 in central


	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.6 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central


	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central


	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 1069ms :: artifacts dl 18ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	commons-logging#commons-logging;1.1.3 from central in [default]
	org.apache.commons#commons-pool2;2.11.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.3.4 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.3.4 from central in [default]
	org.apache.kafka#kafka-clients;3.4.1 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.6 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.6 from central in [default]
	org.lz4#lz4-java;1.8.0 from central in [default]
	org.slf4j#slf4j-api;2.0.7 from central in [default]
	org.xerial.snappy#snappy-java;1.1.10.5 from central in [default]
	---------------------------------------------------------------------


26/09/23 12:24:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark: 3.5.6


## 1. Подготовка CSV и независимых выборок

Убираю пустые и повторяющиеся тексты до разделения. Порядок определяет SHA-256 нормализованного текста: результат воспроизводим и одинаковые комментарии не попадают в обе группы. 6000 комментариев для обучения, следующие 1000 — validation для выбора модели, ещё 1000 — независимый test и отправка в Kafka. Классы не балансирую искусственно; отдельно показываю их доли.

In [2]:
source = pd.read_parquet('data/civil_comments.parquet', columns=['text','toxicity'])
source['text'] = source['text'].fillna('').str.strip()
source = source[source.text.str.len().between(1,10000)].copy()
source['normalized'] = source.text.str.lower().str.replace(r'\s+', ' ', regex=True)
source = source.drop_duplicates('normalized')
source['event_id'] = source.normalized.map(lambda x: hashlib.sha256(x.encode()).hexdigest())
source['label'] = (source.toxicity >= .5).astype(int)
source = source.sort_values('event_id')
train_pd = source.iloc[:6000][['event_id','text','label']].copy()
valid_pd = source.iloc[6000:7000][['event_id','text','label']].copy()
test_pd = source.iloc[7000:8000][['event_id','text','label']].copy()
assert set(train_pd.event_id).isdisjoint(test_pd.event_id)
assert train_pd.label.nunique() == test_pd.label.nunique() == 2
train_pd.to_csv('data/train.csv',index=False)
test_pd.to_csv('data/comments.csv',index=False)
display(pd.DataFrame({'rows':[len(train_pd),len(test_pd)],
    'toxic_share':[train_pd.label.mean(),test_pd.label.mean()]},index=['train','held-out']))
train = spark.createDataFrame(train_pd).withColumn('label',F.col('label').cast('double'))
test = spark.createDataFrame(test_pd).withColumn('label',F.col('label').cast('double'))
valid = spark.createDataFrame(valid_pd).withColumn('label',F.col('label').cast('double'))

,rows,toxic_share
train,6000,0.077833
held-out,1000,0.090000


## 2. Обучение Spark ML

Tokenizer → HashingTF → IDF → LogisticRegression. IDF и классификатор обучаются только на train. При редком токсичном классе одной accuracy недостаточно: считаю precision, recall, F1, ROC-AUC и PR-AUC, сравниваю с ответом «всё нетоксично». По F1 токсичного класса на validation выбираю обычную модель или модель с весами классов, рассчитанными только по train. Порог 0.5 фиксирован. Итоговый test не участвует в выборе.

In [3]:
pipeline = Pipeline(stages=[
    RegexTokenizer(inputCol='text',outputCol='words',pattern='\\W+',minTokenLength=2),
    HashingTF(inputCol='words',outputCol='raw_features',numFeatures=16384),
    IDF(inputCol='raw_features',outputCol='features',minDocFreq=2),
    LogisticRegression(featuresCol='features',labelCol='label',maxIter=20,regParam=.1)])
baseline_model = pipeline.fit(train)
# Inverse-frequency weights use only training labels.
positive_weight = float((train_pd.label == 0).sum() / (train_pd.label == 1).sum())
weighted_train = train.withColumn('class_weight',F.when(F.col('label')==1,positive_weight).otherwise(1.0))
pipeline.getStages()[-1].setWeightCol('class_weight')
balanced_model = pipeline.fit(weighted_train)
def positive_f1(candidate, frame):
    counts = {(int(r.label),int(r.prediction)):r['count'] for r in candidate.transform(frame).groupBy('label','prediction').count().collect()}
    tp,fp,fn = counts.get((1,1),0),counts.get((0,1),0),counts.get((1,0),0)
    return 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0
validation_scores = {'unweighted':positive_f1(baseline_model,valid),'balanced':positive_f1(balanced_model,valid)}
selected = max(validation_scores,key=validation_scores.get)
model = balanced_model if selected=='balanced' else baseline_model
print('Validation F1:',validation_scores,'Selected:',selected)
model.write().overwrite().save('work/toxicity_model')
predictions = model.transform(test).cache()
cm = {(int(r.label),int(r.prediction)):r['count'] for r in predictions.groupBy('label','prediction').count().collect()}
tn,fp,fn,tp = [cm.get(k,0) for k in [(0,0),(0,1),(1,0),(1,1)]]
precision = tp/(tp+fp) if tp+fp else 0
recall = tp/(tp+fn) if tp+fn else 0
metrics = {'train_rows':len(train_pd),'test_rows':len(test_pd),
    'accuracy':(tp+tn)/len(test_pd),'baseline_accuracy':1-test_pd.label.mean(),
    'precision':precision,'recall':recall,'f1':2*precision*recall/(precision+recall) if precision+recall else 0,
    'roc_auc':BinaryClassificationEvaluator(metricName='areaUnderROC').evaluate(predictions),
    'pr_auc':BinaryClassificationEvaluator(metricName='areaUnderPR').evaluate(predictions),
    'tn':tn,'fp':fp,'fn':fn,'tp':tp, 'selected_model':selected,'validation_f1':validation_scores}
display(pd.Series({k:v for k,v in metrics.items() if not isinstance(v,dict)}).to_frame('value'))
predictions.unpersist()

Validation F1: {'unweighted': 0.05063291139240506, 'balanced': 0.13793103448275862} Selected: balanced


,value
train_rows,6000
test_rows,1000
accuracy,0.904
baseline_accuracy,0.91
precision,0.428571
recall,0.2
f1,0.272727
roc_auc,0.671245
pr_auc,0.282659
tn,886


DataFrame[event_id: string, text: string, label: double, words: array<string>, raw_features: vector, features: vector, rawPrediction: vector, probability: vector, prediction: double]

## 3. CSV → Kafka

Каждая запись содержит ID, текст, исходную метку и время события. Для короткой демонстрации присваиваю последовательные времена с интервалом 1 секунда — это моделирование потока, а не время публикации комментариев. Топик отдельный для каждого запуска, без внешних портов. Отправляю ещё 10 повторов для проверки дедупликации.

In [4]:
broker = os.environ.get('KAFKA_BOOTSTRAP_SERVERS','kafka:9092')
for attempt in range(60):
    try:
        admin = KafkaAdminClient(bootstrap_servers=broker,request_timeout_ms=10000)
        break
    except Exception:
        if attempt == 59: raise
        time.sleep(2)
topic = 'toxic-comments-' + uuid.uuid4().hex[:12]
admin.create_topics([NewTopic(topic,num_partitions=1,replication_factor=1)])
admin.close()
csv_data = pd.read_csv('data/comments.csv')
base_time = datetime(2025,1,1,tzinfo=timezone.utc)
events = [dict(event_id=row.event_id,text=row.text,label=int(row.label),
               event_time=(base_time+timedelta(seconds=i)).isoformat())
          for i,row in enumerate(csv_data.itertuples(index=False))]
producer = KafkaProducer(bootstrap_servers=broker,acks='all',
    value_serializer=lambda value: json.dumps(value).encode('utf-8'))
for event in events + events[:10]:
    producer.send(topic,value=event).get(timeout=30)
producer.flush(); producer.close()
print('Sent:',len(events)+10,'unique:',len(events))

Sent: 1010 unique: 1000


## 4. Streaming → модель → минутные окна → memory sink

Декодирую Kafka value и JSON по явной схеме. Вычисляю длину комментария. Убираю повторы по ID; watermark 1 час ограничивает состояние для этой демонстрации. Memory sink работает в complete mode, хранится в памяти драйвера и предназначен для анализа, а не долговременного хранения. AvailableNow обрабатывает доступный поток и завершает запрос.

In [5]:
schema = T.StructType([
    T.StructField('event_id',T.StringType()),T.StructField('text',T.StringType()),
    T.StructField('label',T.IntegerType()),T.StructField('event_time',T.StringType())])
raw = (spark.readStream.format('kafka').option('kafka.bootstrap.servers',broker)
       .option('subscribe',topic).option('startingOffsets','earliest').load())
parsed = raw.select(F.from_json(F.col('value').cast('string'),schema).alias('data')).select('data.*')
stream = (parsed.withColumn('event_time',F.to_timestamp('event_time'))
    .withColumn('comment_length',F.length('text'))
    .filter(F.col('event_id').isNotNull() & F.col('event_time').isNotNull() & (F.col('comment_length')>0))
    .withWatermark('event_time','1 hour').dropDuplicatesWithinWatermark(['event_id']))
scored = model.transform(stream)
counts = scored.groupBy(F.window('event_time','1 minute'),'prediction').count()
query = (counts.writeStream.format('memory').queryName('toxicity_windows')
    .outputMode('complete').option('checkpointLocation',f'work/checkpoint-{topic}')
    .trigger(availableNow=True).start())
query.awaitTermination(240)
if query.isActive:
    query.stop()
    raise TimeoutError('Streaming query did not finish')
if query.exception(): raise RuntimeError(str(query.exception()))
result = spark.sql('SELECT window.start AS minute, CAST(prediction AS INT) AS toxic, count FROM toxicity_windows ORDER BY minute,toxic').toPandas()
assert int(result['count'].sum()) == len(events), 'Lost messages or duplicate IDs'
metrics['stream_unique_count'] = int(result['count'].sum())
metrics['duplicates_removed'] = 10
display(result.head(10))

,minute,toxic,count
0,2025-01-01 00:00:00,0,59
1,2025-01-01 00:00:00,1,1
2,2025-01-01 00:01:00,0,59
3,2025-01-01 00:01:00,1,1
4,2025-01-01 00:02:00,0,54
5,2025-01-01 00:02:00,1,6
6,2025-01-01 00:03:00,0,59
7,2025-01-01 00:03:00,1,1
8,2025-01-01 00:04:00,0,55
9,2025-01-01 00:04:00,1,5


In [6]:
plot_data = result.pivot(index='minute',columns='toxic',values='count').fillna(0)
plot_data = plot_data.reindex(columns=[0,1],fill_value=0).rename(columns={0:'Non-toxic prediction',1:'Toxic prediction'})
plot_data.index = pd.to_datetime(plot_data.index).strftime('%H:%M')
ax = plot_data.plot.bar(stacked=True,figsize=(11,4),color=['#178c83','#e68b55'])
ax.set(title='Comment predictions by event-time minute (UTC)',xlabel='Minute',ylabel='Comments')
plt.tight_layout()
plt.savefig('evidence/stream_counts.png',dpi=150)
display(ax.figure)
plt.close()
result.to_csv('evidence/window_counts.csv',index=False)
Path('evidence/metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
spark.stop()

<Figure size 1100x400 with 1 Axes>

## Вывод

Проверен полный путь CSV → Kafka → Spark Structured Streaming → обученная модель → оконная агрегация → memory sink → Pandas и график. Проверка количества подтверждает, что 10 повторов не увеличили итоговый счётчик.

Качество модели нужно оценивать по таблице метрик, особенно recall токсичного класса. Небольшая учебная выборка и простой словарный подход не дают оснований использовать модель для автоматической модерации. Для улучшения я бы увеличил обучающую выборку, проверил другие признаки и выбирал порог на отдельной validation-части, сохраняя test для итоговой проверки. Токсичность зависит от контекста; предсказание не является объективной оценкой человека.